In [ ]:
# Import python packages
import streamlit as st
import pandas as pd

from snowflake.snowpark.context import get_active_session
session = get_active_session()
from datetime import datetime, timedelta
from snowflake.ml.registry import Registry
import joblib
from snowflake.ml.modeling.pipeline import Pipeline
import snowflake.ml.modeling.preprocessing as pp
from snowflake.ml.modeling.xgboost import XGBClassifier
from snowflake.snowpark.types import StringType, IntegerType
import snowflake.snowpark.functions as F
from snowflake.snowpark.functions import col, current_date, dateadd, random, floor,current_date, datediff

session.query_tag = {"origin":"sf_sit-is", "name":"mlops_customerchurn", "version":{"major":1, "minor":0}}

import snowflake.snowpark.functions as F
from IPython.display import Markdown, display

solution_prefix = session.get_current_warehouse()
solution_prefix

In [ ]:
-- Create csv format
CREATE FILE FORMAT IF NOT EXISTS CSVFORMAT 
    SKIP_HEADER = 1 
    TYPE = 'CSV';
    
CREATE OR REPLACE STAGE data_stage
    FILE_FORMAT = (TYPE = 'CSV') 
    URL = 's3://sfquickstarts/sfguide_getting_started_with_ml_observability_in_snowflake/mlops_customerchurn.csv';
    
-- Inspect content of stage
LS @data_stage;

In [ ]:
spdf = session.read.options({"field_delimiter": ",",
                                    "field_optionally_enclosed_by": '"',
                                    "infer_schema": True,
                                    "parse_header": True}).csv("@data_stage")

In [ ]:
from snowflake.snowpark.types import DecimalType, FloatType, DoubleType

# Get schema of the DataFrame
schema = spdf.schema.fields

# Identify columns that are of type NUMBER (DecimalType)
num_columns = [col.name for col in schema if isinstance(col.datatype, DecimalType)]

# Convert columns to FLOAT
for col in num_columns:
    spdf = spdf.with_column(col, spdf[col].cast(DoubleType()))

In [ ]:
from snowflake.snowpark import Session
from snowflake.snowpark.functions import col, current_date, dateadd, to_date,lit

# Step 1: Get today's date
todays_date = datetime.now()

latest_date = max(spdf.select('TRANSACTIONTIMESTAMP').collect())[0]

# Step 3: Calculate the difference in days
diff_days = (todays_date - latest_date).days - 1

df = spdf.with_column(
    "TRANSACTIONTIMESTAMP", 
    dateadd("day", lit(diff_days), col("TRANSACTIONTIMESTAMP"))
)


df = df.with_column(
    "CREDITSCORE", col("CREDITSCORE").cast("float")
)
df = df.with_column(
    "PREDICTED_CHURN", F.lit(9999)
)
df.show()

df.write.mode("overwrite").save_as_table("CUSTOMERS")

In [ ]:
spdf= df.drop('ROWNUMBER')

Splitting the data into training and testing sets

In [ ]:
num_cols = ['ESTIMATEDSALARY', 'BALANCE', 'CREDITSCORE','AGE','TENURE','DEBTTOINCOME']
output_cols=['EstimatedSalary_SS', 'Balance_SS', 'CreditScore_SS','Age_SS','Tenure_SS','debttoincome_SS']

cat_cols = ['HasCrCard', 'IsActiveMember', 'Geography','Gender', 'NumOfProducts']
string_columns = ['GEOGRAPHY', 'GENDER']
string_columns_oe = ['GEOGRAPHY_oe', 'GENDER_oe']
preprocessing_pipeline = Pipeline(
    steps=[
            (
                "OE",
                pp.OrdinalEncoder(
                    input_cols=string_columns,
                    output_cols=string_columns_oe,
                    drop_input_cols= False,
                )
                
            ),
            (
                "MMS",
                pp.MinMaxScaler(
                    clip=True,
                    input_cols=num_cols,
                    output_cols=output_cols,
                    drop_input_cols= False,
                )
            )
    ]
)

PIPELINE_FILE = '/tmp/preprocessing_pipeline.joblib'
joblib.dump(preprocessing_pipeline, PIPELINE_FILE) # We are just pickling it locally first
training, testing = spdf.random_split(weights=[0.8, 0.2], seed=111)
training_spdf = preprocessing_pipeline.fit(training).transform(training)
testing_spdf=preprocessing_pipeline.fit(testing).transform(testing)

In [ ]:
session.sql("CREATE or replace stage ML_STAGE").collect()
session.file.put(PIPELINE_FILE, "@ML_STAGE", overwrite=True)

In [ ]:
ls @ML_STAGE

XGBClassifier model and train

In [ ]:
num_cols = ['EstimatedSalary', 'Balance', 'CreditScore','Age','Tenure','debttoincome']

cat_cols = ['HasCrCard', 'IsActiveMember', 'GEOGRAPHY','GENDER', 'NumOfProducts']
Target = ["EXITED"]

feature_names_input = [c for c in training_spdf.columns if c not in ["EXITED", "TRANSACTIONTIMESTAMP", "CUSTOMERID","ESTIMATEDSALARY", "BALANCE", "CREDITSCORE","AGE","TENURE","DEBTTOINCOME","GEOGRAPHY","GENDER","PREDICTED_CHURN"]]


training_spdf = training_spdf.with_column(
    "CREDITSCORE_SS", col("CREDITSCORE_SS").cast("float")
)
output_label = ["PREDICTED_CHURN"]
# Initialize a XGBClassifier object with input, label, and output column names
model = XGBClassifier(
    input_cols=feature_names_input,
    label_cols=Target,
    output_cols=output_label
    
)

# Train the classifier model using the training set
_ = model.fit(training_spdf)

Initalize Model Registry

In [ ]:
from snowflake.ml.registry import Registry
from snowflake.ml.model import type_hints

reg = Registry(session=session)

MODEL_NAME = "QS_CustomerChurn_classifier"
MODEL_VERSION = "v1_wh"

mv = reg.log_model(model,
                   model_name=MODEL_NAME,
                   version_name=MODEL_VERSION,
                   sample_input_data=training_spdf,
                   target_platforms=["WAREHOUSE"],
                   task=type_hints.Task.TABULAR_BINARY_CLASSIFICATION)
reg.show_models()

Ongoing inference

In [ ]:
from snowflake.ml.modeling.pipeline import Pipeline
import snowflake.ml.modeling.preprocessing as pp
import snowflake.snowpark.functions as F

def inference(table_name, modelname, modelversion) -> str:
    reg = Registry(session=session)
    m = reg.get_model(modelname)
    mv = m.version(modelversion)
    
    # Load preprocessing pipeline from a file
    session.file.get('@ML_STAGE/preprocessing_pipeline.joblib.gz', '/tmp')
    pipeline_file = '/tmp/preprocessing_pipeline.joblib.gz'
    
    
    preprocessing_pipeline = joblib.load(pipeline_file)
    
    df = session.table(table_name)
    
    # Apply preprocessing
    testing_spdf = preprocessing_pipeline.fit(df).transform(df)
    testing_spdf = testing_spdf.with_column(
    "CREDITSCORE_SS", col("CREDITSCORE_SS").cast("float")
)
    # Perform prediction
    results = mv.run(testing_spdf, function_name="predict")
    results =results.drop("CREDITSCORE_SS", "BALANCE_SS", "DEBTTOINCOME_SS", "TENURE_SS", "AGE_SS", "ESTIMATEDSALARY_SS", "GENDER_OE", "GEOGRAPHY_OE")
    #results.write.save_as_table("customer_churn", mode="overwrite")
    results.create_or_replace_temp_view("results_temp")
    update_statement = f"""
    UPDATE {table_name} t
    SET PREDICTED_CHURN = r.PREDICTED_CHURN
    FROM results_temp r
    WHERE t.CUSTOMERID = r.CUSTOMERID
    AND t.TRANSACTIONTIMESTAMP=r.TRANSACTIONTIMESTAMP;
"""

    # Execute the merge statement
    session.sql(update_statement).collect()
        
    return "Success"

In [ ]:
MODEL_NAME = "QS_CustomerChurn_classifier"
MODEL_VERSION = "v1_wh"

mv = reg.get_model(MODEL_NAME).version(MODEL_VERSION)
testing_spdf = testing_spdf.with_column(
    "CREDITSCORE_SS", col("CREDITSCORE_SS").cast("float")
)
results = mv.run(testing_spdf, function_name="predict")
results

DATA DRIFT AND OBSERVABILITY

In [ ]:
-- Create csv format
CREATE FILE FORMAT IF NOT EXISTS CSVFORMAT 
    SKIP_HEADER = 1 
    TYPE = 'CSV';
    
CREATE OR REPLACE STAGE data_stage
    FILE_FORMAT = (TYPE = 'CSV') 
    URL = 's3://sfquickstarts/sfguide_getting_started_with_ml_observability_in_snowflake/CUSTOMERS_DRIFTED.csv';
    
-- Inspect content of stage
LS @data_stage;

In [ ]:
spdf = session.read.options({"field_delimiter": ",",
                                    "field_optionally_enclosed_by": '"',
                                    "infer_schema": True,
                                    "parse_header": True}).csv("@data_stage")

from snowflake.snowpark.types import DecimalType, FloatType

# Get schema of the DataFrame
schema = spdf.schema.fields

# Identify columns that are of type NUMBER (DecimalType)
num_columns = [col.name for col in schema if isinstance(col.datatype, DecimalType)]

# Convert columns to FLOAT
for col in num_columns:
    spdf = spdf.with_column(col, spdf[col].cast(FloatType()))


from snowflake.snowpark.functions import col, current_date, dateadd, to_date, lit,to_timestamp
from datetime import datetime

# Step 1: Get today's date
todays_date = datetime.now()

# Ensure TRANSACTIONTIMESTAMP is stored as a string first
spdf = spdf.with_column("TRANSACTIONTIMESTAMP", col("TRANSACTIONTIMESTAMP").cast("string"))

# Get the latest date from the dataset
latest_date_str = max(spdf.select('TRANSACTIONTIMESTAMP').collect())[0]

# Convert latest_date to datetime
latest_date = datetime.strptime(latest_date_str, '%m/%d/%y %H:%M')

# Step 3: Calculate the difference in days
diff_days = (todays_date - latest_date).days - 1

# Apply date adjustment
df = spdf.with_column(
    "TRANSACTIONTIMESTAMP",
    dateadd("day", lit(diff_days), to_timestamp(col("TRANSACTIONTIMESTAMP"), 'MM/DD/YY HH24:MI'))
)

# Cast CREDIT SCORE to float
df = df.with_column("CREDITSCORE", col("CREDITSCORE").cast("float"))

# Add PREDICTED_CHURN column
spdf_drift = df.with_column("PREDICTED_CHURN", lit(9999))

spdf_drift.show()

In [ ]:
current_columns = spdf_drift.columns 
new_columns = [col.strip('"') for col in current_columns] 

spdf_drift = spdf_drift.select([spdf_drift[col].alias(new_col) for col, new_col in zip(current_columns, new_columns)])
spdf_drift = spdf_drift.with_column(
    "CREDITSCORE", col("CREDITSCORE").cast("float")
)
spdf_drift.write.mode("overwrite").save_as_table("CUSTOMERS_DRIFTED")
spdf_drift.write.mode("overwrite").save_as_table("CUSTOMERS_EVAL")

Enable Monitoring

In [ ]:
'''
reg = Registry(session=session,options={"enable_monitoring": True})
modelname='QS_CustomerChurn_classifier'
modelversion='v1'
m = reg.get_model(modelname)
mv = m.version(modelversion)

# Fetch model version that will be monitored
model_version = mv

from snowflake.ml.monitoring.entities.model_monitor_config import ModelMonitorConfig, ModelMonitorSourceConfig
source_config = ModelMonitorSourceConfig(
    source="CUSTOMERS_DRIFTED",
    baseline="CUSTOMERS",
    timestamp_column="TRANSACTIONTIMESTAMP",
    prediction_class_columns=["PREDICTED_CHURN"],
    actual_class_columns=["EXITED"],
    id_columns=["CUSTOMERID"],
)

# Set up config for ModelMonitor.
model_monitor_config = ModelMonitorConfig(
    model_version=model_version,
    model_function_name="predict",
    background_compute_warehouse_name="ml_wh",
)

# Add a new ModelMonitor
model_monitor = reg.add_monitor(
    name=f"CHURN_MODEL_MONITOR", 
    source_config=source_config,
    model_monitor_config=model_monitor_config,
)
model_monitor
'''

SQL for model monitor

In [ ]:
query = f"""
CREATE OR REPLACE MODEL MONITOR CHURN_MODEL_MONITOR
WITH
    MODEL=QS_CustomerChurn_classifier
    VERSION=v1_wh
    FUNCTION=predict
    SOURCE=CUSTOMERS_DRIFTED
    BASELINE=CUSTOMERS
    TIMESTAMP_COLUMN=TRANSACTIONTIMESTAMP
    PREDICTION_CLASS_COLUMNS=(PREDICTED_CHURN)  
    ACTUAL_CLASS_COLUMNS=(EXITED)
    ID_COLUMNS=(CUSTOMERID)
    SEGMENT_COLUMNS = (GEOGRAPHY)
    WAREHOUSE=COMPUTE_WH
    REFRESH_INTERVAL='1 min'
    AGGREGATION_WINDOW='1 day';
"""
session.sql(query).collect()

In [ ]:
status= inference('CUSTOMERS_DRIFTED','QS_CUSTOMERCHURN_CLASSIFIER', 'v1_wh');

In [ ]:
-- Create csv format
CREATE FILE FORMAT IF NOT EXISTS CSVFORMAT 
    SKIP_HEADER = 1 
    TYPE = 'CSV';
    
CREATE OR REPLACE STAGE data_stage
    FILE_FORMAT = (TYPE = 'CSV') 
    URL = 's3://sfquickstarts/sfguide_getting_started_with_ml_observability_in_snowflake/CUSTOMERS_TRAINING.csv';
    
-- Inspect content of stage
LS @data_stage;

retrain model based on new data

In [ ]:
spdf = session.read.options({"field_delimiter": ",",
                                    "field_optionally_enclosed_by": '"',
                                    "infer_schema": True,
                                    "parse_header": True}).csv("@data_stage")

from snowflake.snowpark.types import DecimalType, FloatType

# Get schema of the DataFrame
schema = spdf.schema.fields

# Identify columns that are of type NUMBER (DecimalType)
num_columns = [col.name for col in schema if isinstance(col.datatype, DecimalType)]

# Convert columns to FLOAT
for col in num_columns:
    spdf = spdf.with_column(col, spdf[col].cast(FloatType()))


from snowflake.snowpark.functions import col, current_date, dateadd, to_date, lit,to_timestamp
from datetime import datetime

# Step 1: Get today's date
todays_date = datetime.now()

# Ensure TRANSACTIONTIMESTAMP is stored as a string first
spdf = spdf.with_column("TRANSACTIONTIMESTAMP", col("TRANSACTIONTIMESTAMP").cast("string"))

# Get the latest date from the dataset
latest_date_str = max(spdf.select('TRANSACTIONTIMESTAMP').collect())[0]

# Convert latest_date to datetime
latest_date = datetime.strptime(latest_date_str, '%m/%d/%y %H:%M')

# Step 3: Calculate the difference in days
diff_days = (todays_date - latest_date).days - 1

# Apply date adjustment
df = spdf.with_column(
    "TRANSACTIONTIMESTAMP",
    dateadd("day", lit(diff_days), to_timestamp(col("TRANSACTIONTIMESTAMP"), 'MM/DD/YY HH24:MI'))
)

# Cast CREDIT SCORE to float
df = df.with_column("CREDITSCORE", col("CREDITSCORE").cast("float"))

# Add PREDICTED_CHURN column
spdf_drift = df.with_column("PREDICTED_CHURN", lit(9999))

spdf_drift.show()

current_columns = spdf_drift.columns 
new_columns = [col.strip('"') for col in current_columns] 

spdf_drift = spdf_drift.select([spdf_drift[col].alias(new_col) for col, new_col in zip(current_columns, new_columns)])
spdf_drift = spdf_drift.with_column(
    "CREDITSCORE", col("CREDITSCORE").cast("float")
)


spdf_drift.write.mode("overwrite").save_as_table("CUSTOMERS_TRAINING")

In [ ]:
# Load preprocessing pipeline from a file
session.file.get('@ML_STAGE/preprocessing_pipeline.joblib.gz', '/tmp')
pipeline_file = '/tmp/preprocessing_pipeline.joblib.gz'


preprocessing_pipeline = joblib.load(pipeline_file)

# Apply preprocessing
training_spdf = preprocessing_pipeline.fit(spdf_drift).transform(spdf_drift)
training_spdf = training_spdf.with_column(
"CREDITSCORE_SS", col("CREDITSCORE_SS").cast("float")
)

num_cols = ['EstimatedSalary', 'Balance', 'CreditScore','Age','Tenure','debttoincome']

cat_cols = ['HasCrCard', 'IsActiveMember', 'GEOGRAPHY','GENDER', 'NumOfProducts']
Target = ["EXITED"]

feature_names_input = [c for c in training_spdf.columns if c not in ["EXITED", "TRANSACTIONTIMESTAMP", "CUSTOMERID","ESTIMATEDSALARY", "BALANCE", "CREDITSCORE","AGE","TENURE","DEBTTOINCOME","GEOGRAPHY","GENDER","PREDICTED_CHURN","DATASET_TYPE"]]

output_label = ["PREDICTED_CHURN"]
# Initialize a XGBClassifier object with input, label, and output column names
model = XGBClassifier(
    input_cols=feature_names_input,
    label_cols=Target,
    output_cols=output_label
    
)

# Train the classifier model using the training set
_ = model.fit(training_spdf)

Log the model as a new version V2

In [ ]:
from snowflake.ml.registry import Registry
from snowflake.ml.model import type_hints

reg = Registry(session=session)

MODEL_NAME = "QS_CustomerChurn_classifier"
MODEL_VERSION = "v2"

mv = reg.log_model(model,
                   model_name=MODEL_NAME,
                   version_name=MODEL_VERSION,
                   sample_input_data=training_spdf,
                   target_platforms=["WAREHOUSE"],
                   task=type_hints.Task.TABULAR_BINARY_CLASSIFICATION)
reg.show_models()

In [ ]:
query = f"""
CREATE OR REPLACE MODEL MONITOR CHURN_MODEL_MONITOR_NEW
WITH
    MODEL=QS_CustomerChurn_classifier
    VERSION=v2
    FUNCTION=predict
    SOURCE=CUSTOMERS_EVAL
    BASELINE=CUSTOMERS_TRAINING
    TIMESTAMP_COLUMN=TRANSACTIONTIMESTAMP
    PREDICTION_CLASS_COLUMNS=(PREDICTED_CHURN)  
    ACTUAL_CLASS_COLUMNS=(EXITED)
    ID_COLUMNS=(CUSTOMERID)
    SEGMENT_COLUMNS = (GEOGRAPHY)
    WAREHOUSE=COMPUTE_WH
    REFRESH_INTERVAL='1 min'
    AGGREGATION_WINDOW='1 day';
"""
session.sql(query).collect()

In [ ]:
status= inference('CUSTOMERS_EVAL','QS_CUSTOMERCHURN_CLASSIFIER', 'v2');

In [ ]:
SELECT * FROM 
TABLE(
MODEL_MONITOR_STAT_METRIC(
'CHURN_MODEL_MONITOR', 'COUNT', 'PREDICTED_CHURN', '1 DAY', 
DATEADD('day', -7, CURRENT_TIMESTAMP()), CURRENT_TIMESTAMP())
) as a
JOIN (SELECT * FROM 
TABLE(
MODEL_MONITOR_STAT_METRIC(
'CHURN_MODEL_MONITOR_NEW', 'COUNT', 'PREDICTED_CHURN', '1 DAY', 
DATEADD('day', -7, CURRENT_TIMESTAMP()), CURRENT_TIMESTAMP())
)) as b ON a.EVENT_TIMESTAMP = b.EVENT_TIMESTAMP;

In [ ]:
SELECT * FROM TABLE(MODEL_MONITOR_DRIFT_METRIC(
'CHURN_MODEL_MONITOR_NEW', 'DIFFERENCE_OF_MEANS', 'PREDICTED_CHURN', '1 DAY', 
DATEADD('day', -7, CURRENT_TIMESTAMP()), CURRENT_TIMESTAMP()))

In [ ]:
SELECT 
    'v1_wh' as model_version,
    * 
FROM TABLE(MODEL_MONITOR_PERFORMANCE_METRIC(
    'CHURN_MODEL_MONITOR', 'PRECISION', '1 DAY', 
    DATEADD('day', -7, CURRENT_TIMESTAMP()), CURRENT_TIMESTAMP()
))
UNION ALL
SELECT 
    'v2' as model_version,
    * 
FROM TABLE(MODEL_MONITOR_PERFORMANCE_METRIC(
    'CHURN_MODEL_MONITOR_NEW', 'PRECISION', '1 DAY', 
    DATEADD('day', -7, CURRENT_TIMESTAMP()), CURRENT_TIMESTAMP()
))
ORDER BY EVENT_TIMESTAMP, model_version;

In [ ]:
query=f'''CREATE or replace TABLE TEST_NOTIFICATION(
    notification varchar (100),
    created_at timestamp
);'''

session.sql(query).collect()

In [ ]:
CREATE OR REPLACE ALERT high_drift_alert
    WAREHOUSE = COMPUTE_WH
    SCHEDULE = '60 minutes'
    IF ( EXISTS (SELECT * FROM TABLE(MODEL_MONITOR_DRIFT_METRIC(
    'CHURN_MODEL_MONITOR', 'DIFFERENCE_OF_MEANS', 'PREDICTED_CHURN', '1 MONTH', TO_TIMESTAMP_TZ('2024-01-01'), TO_TIMESTAMP_TZ('2025-02-04')))))
    THEN
        INSERT INTO TEST_NOTIFICATION (notification, created_at) VALUES ('ALERT',(SELECT CURRENT_TIMESTAMP));